In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import torch
import time
import math
import torch.nn.functional as F

B = 8          # batch size
NH = 16        # number of heads
N = 1024       # sequence length (ONLY this as asked)
D = 64         # head dimension

M=32*1024
Bc = int(min(M // (4 * D), N))
Br = int(min(Bc, D))
Tc = (N + Bc - 1) // Bc
Tr = (N + Br - 1) // Br


device="cuda"

Q=torch.randn(B,NH,N,D,device=device,dtype=torch.float32)
K=torch.randn(B,NH,N,D,device=device,dtype=torch.float32)
V=torch.randn(B,NH,N,D,device=device,dtype=torch.float32)

def standard_attention(Q,K,V):
    scores=torch.matmul(Q,K.transpose(-1,-2))
    scores=scores/math.sqrt(D)
    attn=torch.softmax(scores,dim=-1)
    O=torch.matmul(attn,V)
    return O

def torch_flash_attention(Q,K,V):
    O=F.scaled_dot_product_attention(Q,K,V,dropout_p=0.0,attn_mask=None,is_causal=False)
    return O

def python_flash_attention(Q,K,V):
    O=torch.zeros_like(Q)
    for b in range(B):
        for h in range(NH):
            Qbh=Q[b,h]
            Kbh=K[b,h]
            Vbh=V[b,h]
            Obh=torch.zeros_like(Qbh)

            M = torch.full((N,1), -float('inf'), device=Q.device, dtype=Q.dtype)
            L = torch.zeros((N,1), device=Q.device, dtype=Q.dtype)
            
            for j in range(Tc):
                Kij=Kbh[j*Bc:(j+1)*Bc]
                Vij=Vbh[j*Bc:(j+1)*Bc]
                
                for i in range(Tr):
                    Qij=Qbh[i*Br:(i+1)*Br]
                    O_i=Obh[i*Br:(i+1)*Br]
                    Sij=torch.matmul(Qij,Kij.T)
                    Sij=Sij/math.sqrt(D)

                    m_i=torch.max(Sij,dim=1,keepdim=True).values
                    Pij=torch.exp(Sij-m_i)
                    lij=torch.sum(Pij,dim=1,keepdim=True)

                    m_old=M[i*Br:(i+1)*Br]
                    l_old=L[i*Br:(i+1)*Br]
                    
                    
                    m_new=torch.maximum(m_i,m_old)
                    alpha=torch.exp(m_old-m_new)
                    beta=torch.exp(m_i-m_new)
                    l_new=alpha*l_old+beta*lij
                    
                    O_i = (alpha * l_old * O_i + beta * (Pij @ Vij)) / l_new
                
                    Obh[i*Br:(i+1)*Br]=O_i
                    M[i*Br:(i+1)*Br]=m_new
                    L[i*Br:(i+1)*Br]=l_new
            O[b,h]=Obh
    
    return O
                    

torch.cuda.synchronize()

iters = 1
start = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    O = standard_attention(Q, K, V)
    torch.cuda.synchronize()

end = time.time()

avg_time_ms = (end - start) * 1000 / iters

print(f"Average Runtime for N=1024 under standard attention: {avg_time_ms:.3f} ms")
print("Output shape:", O.shape)
print()

torch.cuda.synchronize()

start1 = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    Ot = torch_flash_attention(Q, K, V)
    torch.cuda.synchronize()

end1 = time.time()

avg_time_ms1 = (end1 - start1) * 1000 / iters

print(f"Average Runtime for N=1024 under flash attention (torch): {avg_time_ms1:.3f} ms")
print("Output shape:", Ot.shape)
print()

torch.cuda.synchronize()

start2 = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    Ot2 = python_flash_attention(Q, K, V)
    torch.cuda.synchronize()

end2 = time.time()

avg_time_ms2 = (end2 - start2) * 1000 / iters

print(f"Average Runtime for N=1024 under flash attention (python1): {avg_time_ms2:.3f} ms")
print("Output shape:", Ot2.shape)
print()

torch.cuda.synchronize()

max_diff = (O - Ot).abs().max() 
print("Max absolute difference(torch vs pytorch):", max_diff.item())
max_diff2 = (O - Ot2).abs().max()
print("Max absolute difference(python vs torch):", max_diff2.item())
max_diff3 = (Ot - Ot2).abs().max()
print("Max absolute difference(pytorch vs python):", max_diff3.item())


Average Runtime for N=1024 under standard attention: 41.887 ms
Output shape: torch.Size([8, 16, 1024, 64])

Average Runtime for N=1024 under flash attention (torch): 34.953 ms
Output shape: torch.Size([8, 16, 1024, 64])

Average Runtime for N=1024 under flash attention (python1): 5515.181 ms
Output shape: torch.Size([8, 16, 1024, 64])

Max absolute difference(torch vs pytorch): 1.0728836059570312e-06
Max absolute difference(python vs torch): 7.748603820800781e-07
Max absolute difference(pytorch vs python): 8.344650268554688e-07


In [12]:
import torch
import time
import math
import torch.nn.functional as F

B = 8          # batch size
NH = 16        # number of heads
N = 1024       # sequence length (ONLY this as asked)
D = 64         # head dimension

M=32*1024
Bc = int(min(M // (4 * D), N))
Br = int(min(Bc, D))
Tc = (N + Bc - 1) // Bc
Tr = (N + Br - 1) // Br


device="cuda"

Q=torch.randn(B,NH,N,D,device=device,dtype=torch.float16)
K=torch.randn(B,NH,N,D,device=device,dtype=torch.float16)
V=torch.randn(B,NH,N,D,device=device,dtype=torch.float16)

def standard_attention(Q, K, V):
    scores=torch.matmul(Q,K.transpose(-1,-2))
    scores=scores/math.sqrt(D)
    attn=torch.softmax(scores,dim=-1)
    O=torch.matmul(attn,V)
    return O

def torch_flash_attention(Q,K,V):
    O=F.scaled_dot_product_attention(Q,K,V,attn_mask=None,dropout_p=0.0,is_causal=False)
    return O

def python_flash_attention(Q, K, V):
    O=torch.zeros_like(Q)
    for b in range(B):
        for h in range(NH):
            Qbh=Q[b,h]
            Kbh=K[b,h]
            Vbh=V[b,h]
            Obh=O[b,h]
            M=torch.full((N,1),-float('inf'),device=Q.device,dtype=Q.dtype)
            L=torch.zeros((N,1),device=Q.device,dtype=Q.dtype)
            for j in range(Tc):
                Kj=Kbh[j*Bc:(j+1)*Bc]
                Vj=Vbh[j*Bc:(j+1)*Bc]
                for i in range(Tr):
                    Qi=Qbh[i*Br:(i+1)*Br]

                    Sij=torch.matmul(Qi,Kj.T)
                    Sij=Sij/math.sqrt(D)

                    m_i=torch.max(Sij,dim=1,keepdim=True).values
                    Pij=torch.exp(Sij-m_i)
                    l_i=torch.sum(Pij,dim=1,keepdim=True)

                    m_old=M[i*Br:(i+1)*Br]
                    l_old=L[i*Br:(i+1)*Br]

                    m_new=torch.max(m_old,m_i)
                    alpha=torch.exp(m_old-m_new)
                    beta=torch.exp(m_i-m_new)

                    O_old=Obh[i*Br:(i+1)*Br]
                    l_new=alpha*l_old+beta*l_i
                    O_i=(alpha*l_old*O_old+beta*(Pij@Vj))/l_new

                    Obh[i*Br:(i+1)*Br]=O_i
                    L[i*Br:(i+1)*Br]=l_new
                    M[i*Br:(i+1)*Br]=m_new
            O[b,h]=Obh
    return O

def python_flash_attention(Q, K, V):
            O=torch.zeros_like(Q)
            M=torch.full((B,NH,N,1),-float('inf'),device=Q.device,dtype=Q.dtype)
            L=torch.zeros((B,NH,N,1),device=Q.device,dtype=Q.dtype)
            for j in range(Tc):
                Kj=K[:,:,j*Bc:(j+1)*Bc]
                Vj=V[:,:,j*Bc:(j+1)*Bc]
                for i in range(Tr):
                    Qi=Q[:,:,i*Br:(i+1)*Br]

                    Sij=torch.matmul(Qi,Kj.transpose(-1,-2))
                    Sij=Sij/math.sqrt(D)

                    m_i=torch.max(Sij,dim=-1,keepdim=True).values
                    Pij=torch.exp(Sij-m_i)
                    l_i=torch.sum(Pij,dim=-1,keepdim=True)

                    m_old=M[:,:,i*Br:(i+1)*Br]
                    l_old=L[:,:,i*Br:(i+1)*Br]

                    m_new=torch.max(m_old,m_i)
                    alpha=torch.exp(m_old-m_new)
                    beta=torch.exp(m_i-m_new)

                    O_old=O[:,:,i*Br:(i+1)*Br]
                    l_new=alpha*l_old+beta*l_i
                    O_i=(alpha*l_old*O_old+beta*(Pij@Vj))/l_new

                    O[:,:,i*Br:(i+1)*Br]=O_i
                    L[:,:,i*Br:(i+1)*Br]=l_new
                    M[:,:,i*Br:(i+1)*Br]=m_new
            return O


torch.cuda.synchronize()

iters = 1
start = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    O = standard_attention(Q, K, V)
    torch.cuda.synchronize()

end = time.time()

avg_time_ms = (end - start) * 1000 / iters

print(f"Average Runtime for N=1024 under standard attention: {avg_time_ms:.3f} ms")
print("Output shape:", O.shape)
print()

torch.cuda.synchronize()

start1 = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    Ot = torch_flash_attention(Q, K, V)
    torch.cuda.synchronize()

end1 = time.time()

avg_time_ms1 = (end1 - start1) * 1000 / iters

print(f"Average Runtime for N=1024 under flash attention (torch): {avg_time_ms1:.3f} ms")
print("Output shape:", Ot.shape)
print()

torch.cuda.synchronize()

start2 = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    Ot2 = python_flash_attention(Q, K, V)
    torch.cuda.synchronize()

end2 = time.time()

avg_time_ms2 = (end2 - start2) * 1000 / iters

print(f"Average Runtime for N=1024 under flash attention (python1): {avg_time_ms2:.3f} ms")
print("Output shape:", Ot2.shape)
print()

torch.cuda.synchronize()

max_diff = (O - Ot).abs().max() 
print("Max absolute difference(torch vs pytorch):", max_diff.item())
max_diff2 = (O - Ot2).abs().max()
print("Max absolute difference(python vs torch):", max_diff2.item())
max_diff3 = (Ot - Ot2).abs().max()
print("Max absolute difference(pytorch vs python):", max_diff3.item())


Average Runtime for N=1024 under standard attention: 9.602 ms
Output shape: torch.Size([8, 16, 1024, 64])

Average Runtime for N=1024 under flash attention (torch): 5.063 ms
Output shape: torch.Size([8, 16, 1024, 64])

Average Runtime for N=1024 under flash attention (python1): 52.359 ms
Output shape: torch.Size([8, 16, 1024, 64])

Max absolute difference(torch vs pytorch): 0.000732421875
Max absolute difference(python vs torch): 0.000732421875
Max absolute difference(pytorch vs python): 0.0009765625


In [22]:
%%writefile main.cu
#include <iostream>
#include <fstream>
#include <cmath>
#include <cuda.h>
#include <cuda_runtime.h>

#define Br 32
#define Bc 32
#include <random>

void init_random(float* data, size_t size) {
    std::mt19937 gen(42);  // fixed seed for reproducibility
    std::uniform_real_distribution<float> dist(-1.000f, 1.000f);

    for (size_t i = 0; i < size; i++) {
        data[i] = dist(gen);
    }
}

// ====================== IO ======================
void read_bin(const char* fname, float* data, size_t size) {
    std::ifstream f(fname, std::ios::binary);
    f.read((char*)data, size * sizeof(float));
    f.close();
}

void write_bin(const char* fname, float* data, size_t size) {
    std::ofstream f(fname, std::ios::binary);
    f.write((char*)data, size * sizeof(float));
    f.close();
}

// ====================== KERNEL ======================
__global__ void flash_attn(float *Q,float *K,float *V,int B, int NH, int N, int D,float* l, float* m,float* O){
    extern __shared__ float smem[];
    float *Qi=smem;
    float *Kj=Qi+Bc*D;
    float *Vj=Kj+Bc*D;
    float *S=Vj+Bc*D;

    int tx=threadIdx.x;
    int b=blockIdx.x;
    int h=blockIdx.y;

    int Tc = (N + Bc - 1) / Bc;
    int Tr = (N + Br - 1) / Br;
    
    int qkv_base=(b*NH+h)*N*D;
    int lm_base=(b*NH+h)*N;

    float scale = (1.0 / sqrtf(D));

    for(int j=0;j<Tc;j++){
        for(int x=0;x<D;x++){
            int kv_idx=qkv_base+(j*Bc+tx)*D+x;
            Kj[tx*D+x]=K[kv_idx];
            Vj[tx*D+x]=V[kv_idx];
        }
        __syncthreads();
        for(int i=0;i<Tr;i++){
            for(int x=0;x<D;x++){
                int q_idx=qkv_base+(i*Br+tx)*D+x;
                Qi[tx*D+x]=Q[q_idx];
            }
            __syncthreads();
            int lm_idx=lm_base+i*Br+tx;
            float m_prev=m[lm_idx];
            float l_prev=l[lm_idx];
            
            float m_curr= -INFINITY;
            float l_i=0.0;
            for(int k=0;k<Bc;k++){
                float sum=0.0;
                for(int x=0;x<D;x++){
                    float a=Qi[tx*D+x];
                    float b=Kj[k*D+x];
                    sum+=a*b;
                }
                S[tx*Bc+k]=sum*scale;
                if(m_curr<S[tx*Bc+k]){
                    m_curr=S[tx*Bc+k];
                }
            }
            for(int x=0;x<Bc;x++){
                S[tx*Bc+x]=exp(S[tx*Bc+x]-m_curr);
                l_i+=S[tx*Bc+x];
            }
            float m_new=m_prev;
            float l_new=0.0;
            if(m_curr>m_prev) m_new=m_curr;
            float alpha=exp(m_prev-m_new);
            float beta=exp(m_curr-m_new);

            l_new=alpha*l_prev+beta*l_i;

            for(int k=0;k<D;k++){
                float sum=0.0;
                for(int x=0;x<Bc;x++){
                    float a=S[tx*Bc+x];
                    float b=Vj[x*D+k];
                    sum+=a*b;
                }
                int idx=qkv_base+(i * Br + tx) * D + k;
                O[idx]=(alpha*l_prev*O[idx]+beta*sum)/l_new;
            }
            m[lm_idx]=m_new;
            l[lm_idx]=l_new;
        }
        __syncthreads();
    }
}


int main(){
    int B = 8, H = 16, N = 1024, D = 64;

    size_t total = (size_t)B * H * N * D;
    size_t lm_size = (size_t)B * H * N;
    float *O_h,*l_h,*m_h;
    cudaMallocHost(&O_h,total*sizeof(float));
    cudaMallocHost(&l_h,lm_size*sizeof(float));
    cudaMallocHost(&m_h,lm_size*sizeof(float));

    for(int i=0;i<B*H*N*D;i++){
        O_h[i] = 0.0f;
    }
    for(int i=0;i<B*H*N;i++){
        l_h[i] = 0.0f;
        m_h[i] = -INFINITY;
    }

    float *Q = new float[total];
    float *K = new float[total];
    float *V = new float[total];

    init_random(Q, total);
    init_random(K, total);
    init_random(V, total);

    write_bin("Q.bin", Q, total);
    write_bin("K.bin", K, total);
    write_bin("V.bin", V, total);
    
    // also save shape
    std::ofstream fshape("shape.bin", std::ios::binary);
    int shape[4] = {B, H, N, D};
    fshape.write((char*)shape, 4*sizeof(int));
    fshape.close();

    float *d_Q, *d_K, *d_V, *d_O, *d_l, *d_m;

    cudaMalloc(&d_Q, total*sizeof(float));
    cudaMalloc(&d_K, total*sizeof(float));
    cudaMalloc(&d_V, total*sizeof(float));
    cudaMalloc(&d_O, total*sizeof(float));
    cudaMalloc(&d_l, lm_size*sizeof(float));
    cudaMalloc(&d_m, lm_size*sizeof(float));

    cudaMemcpy(d_Q, Q, total*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_K, K, total*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_V, V, total*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_O,O_h,total*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(d_l,l_h,lm_size*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(d_m,m_h,lm_size*sizeof(float),cudaMemcpyHostToDevice);

    dim3 grid(B, H);
    dim3 block(Bc);

    size_t smem = (Br*D + 2*Bc*D + Br*Bc) * sizeof(float);


    flash_attn<<<grid, block, smem>>>(
        d_Q, d_K, d_V,
        B, H, N, D,
        d_l, d_m,
        d_O
    );

    cudaDeviceSynchronize();

    float *O = new float[total];
    cudaMemcpy(O, d_O, total*sizeof(float), cudaMemcpyDeviceToHost);

    write_bin("O_cuda.bin", O, total);

    std::cout << "Done FlashAttention CUDA\n";

    return 0;
}

Writing main.cu


In [23]:
!nvcc main.cu -o main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [24]:
!nvprof ./main

==2832== NVPROF is profiling process 2832, command: ./main
Done FlashAttention CUDA
==2832== Profiling application: ./main
==2832== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   90.89%  497.40ms         1  497.40ms  497.40ms  497.40ms  flash_attn(float*, float*, float*, int, int, int, int, float*, float*, float*)
                    4.67%  25.566ms         1  25.566ms  25.566ms  25.566ms  [CUDA memcpy DtoH]
                    4.44%  24.288ms         6  4.0480ms  46.815us  7.2126ms  [CUDA memcpy HtoD]
      API calls:   51.99%  497.41ms         1  497.41ms  497.41ms  497.41ms  cudaDeviceSynchronize
                   23.10%  220.98ms         3  73.660ms  8.8630us  219.85ms  cudaHostAlloc
                   18.77%  179.61ms         1  179.61ms  179.61ms  179.61ms  cudaLaunchKernel
                    5.61%  53.679ms         7  7.6685ms  58.365us  27.627ms  cudaMemcpy
                    0.42%  4.0639ms       228  1

In [68]:
import torch
import numpy as np
import math
import torch.nn.functional as F

# Load shape
B, H, N, D = np.fromfile("shape.bin", dtype=np.int32)

# Load tensors
Q = torch.tensor(np.fromfile("Q.bin", dtype=np.float32)).reshape(B,H,N,D)
K = torch.tensor(np.fromfile("K.bin", dtype=np.float32)).reshape(B,H,N,D)
V = torch.tensor(np.fromfile("V.bin", dtype=np.float32)).reshape(B,H,N,D)

O_cuda = torch.tensor(np.fromfile("O_cuda.bin", dtype=np.float32)).reshape(B,H,N,D)

# Reference (FlashAttention math)
scores=torch.matmul(Q,K.transpose(-1,-2))
scores=scores/math.sqrt(D)
attn=torch.softmax(scores,dim=-1)
O_ref=torch.matmul(attn,V)

# Compare
diff = torch.abs(O_ref - O_cuda)

print("Max abs diff:", diff.max().item())
print("Mean diff:", diff.mean().item())

Max abs diff: 1.1175870895385742e-07
Mean diff: 7.963901538232676e-09


In [60]:
%%writefile cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <fstream>

#define B 8
#define Nh 16
#define D 64
#define N 1024
#define Br 32
#define Bc 32

void read_file(const char* fname, float* data, int size) {
    std::ifstream file(fname, std::ios::binary);
    if (!file.is_open()) {
        printf("ERROR: cannot open %s\n", fname);  // ✅ catch missing file
        return;
    }
    file.read(reinterpret_cast<char*>(data), size * sizeof(float));
    printf("Read %s: first values = %f %f %f\n",  // ✅ confirm data loaded
           fname, data[0], data[1], data[2]);
    file.close();
 }

__global__ void FlashAttn(const float *Q,const float *K,const float *V,float *m,float *l,float *O){

    int Tr = (N + Br - 1) / Br;
    int Tc = (N + Bc - 1) / Bc;

    int b = blockIdx.x;
    int h = blockIdx.y;
    int tx = threadIdx.x;

    float scale = (1.0 / sqrtf(D));

    extern __shared__ float smem[];
    float *Qi = smem;
    float *Kj = Qi + Bc * D;
    float *Vj = Kj + Bc * D;
    float *S  = Vj + Bc * D;

    int base = (b * Nh + h) * N * D;
    int lm_base = (b * Nh + h) * N;

    for (int j = 0; j < Tc; j++) {
        // # load tiles of K and V
        for(int x = 0; x < D; x++) {
            int index = base + (j * Bc + tx) * D + x;
            Kj[tx * D + x] = K[index];
            Vj[tx * D + x] = V[index];
        }
        __syncthreads();
        for(int i = 0; i < Tr; i++) {

          // # load tile of Q
          for(int x = 0; x < D; x++) {
              int index = base + (i * Br + tx) * D + x;
              Qi[tx * D + x] = Q[index];
          }

          __syncthreads();

          int idx = lm_base + (i * Br) + tx;
          float mprev = m[idx];
          float lprev = l[idx];

          float m_curr = -INFINITY;
          float li = 0.0;
          float m_new = mprev;
          float li_new = 0.0;
          for(int k = 0; k < Bc; k++) {
            float sum = 0;
            for(int x = 0; x < D; x++) {
                float a = Qi[tx * D + x];
                float b = Kj[k * D + x];
                sum += (a * b);
            }
            S[tx * Bc + k] = (sum * scale);

            if(m_curr < S[tx * Bc + k]) {
                m_curr = S[tx * Bc + k];
            }
          }
          for(int x = 0; x < Bc; x++) {
              S[tx * Bc + x] = exp(S[tx * Bc + x] - m_curr);
              li += S[tx * Bc + x];
          }
          if(m_curr > mprev) m_new = m_curr;
          float alpha = exp(mprev - m_new);
          float beta = exp(m_curr - m_new);
          li_new = lprev * alpha + beta * li;

          for(int k = 0; k < D; k++) {
              float sum = 0.0f;
              for(int x = 0; x < Bc; x++) {
                float a = S[tx * Bc + x];
                float b = Vj[x * D + k];
                sum += (a * b);
              }
              int idx = base + (i * Br + tx) * D + k;
              O[idx] = (lprev * O[idx] * alpha + sum * beta) / li_new;
          }
          m[lm_base + (i * Br) + tx] = m_new;
          l[lm_base + (i * Br) + tx] = li_new;
        }
        __syncthreads();
    }
}

int main(){

    //Allocating host memory
    float* Q_h,*K_h,*V_h,*O_h,*l_h,*m_h;
    cudaMallocHost(&Q_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&K_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&V_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&O_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&l_h,B*Nh*N*sizeof(float));
    cudaMallocHost(&m_h,B*Nh*N*sizeof(float));

    // Read from files
    read_file("Q.bin", Q_h, B * Nh * N * D);
    read_file("K.bin", K_h, B * Nh * N * D);
    read_file("V.bin", V_h, B * Nh * N * D);

    //Initializing variables
    for(int i=0;i<B*Nh*N*D;i++){
        O_h[i] = 0;
    }
    for(int i=0;i<B*Nh*N;i++){
        l_h[i] = 0;
        m_h[i] = -INFINITY;
    }


    //Allocating device memory
    float* Q_d,*K_d,*V_d,*O_d,*l_d,*m_d;
    cudaMalloc(&Q_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&K_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&V_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&O_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&l_d,B*Nh*N*sizeof(float));
    cudaMalloc(&m_d,B*Nh*N*sizeof(float));

    //Memory Info
    size_t free_t,total_t;
    size_t used_mem;
    cudaMemGetInfo(&free_t,&total_t);
    used_mem = (total_t - free_t)/(1024*1024);

    printf("free mem: %zu\ntotat mem: %zu\nused mem:%zu MB\n",free_t,total_t,used_mem);

    //Copying to device
    cudaMemcpy(Q_d,Q_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(K_d,K_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(V_d,V_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(O_d,O_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(l_d,l_h,B*Nh*N*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(m_d,m_h,B*Nh*N*sizeof(float),cudaMemcpyHostToDevice);

    //Launching the kernels

    dim3 grid(B,Nh);
    dim3 block(Br);
    size_t smem = (Br*D + 2*Bc*D + Br*Bc)*sizeof(float);

    cudaEvent_t total_start, total_stop;

    cudaEventCreate(&total_start);
    cudaEventCreate(&total_stop);
    cudaEventRecord(total_start);

    FlashAttn<<<grid,block,smem>>>(Q_d,K_d,V_d,m_d,l_d,O_d);

    cudaEventRecord(total_stop);
    cudaEventSynchronize(total_stop);
    float total_time;
    cudaEventElapsedTime(&total_time,total_start,total_stop);

    printf("time:%f\n",total_time/(1000)); //to get seconds for every iteration

    //copy to host
    cudaMemcpy(O_h, O_d, B*Nh*N*D*sizeof(float), cudaMemcpyDeviceToHost);

    std::ofstream out("O_h.bin", std::ios::binary);
    out.write(reinterpret_cast<char*>(O_h), B*Nh*N*D * sizeof(float));
    out.close();


    cudaFree(O_d);
    cudaFree(Q_d);
    cudaFree(K_d);
    cudaFree(V_d);
    cudaFree(m_d);
    cudaFree(l_d);

}

Writing cuda.cu


In [62]:
!nvcc cuda.cu -o cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [63]:
!nvprof ./cuda

==2610== NVPROF is profiling process 2610, command: ./cuda
Read Q.bin: first values = -0.250920 0.593086 0.901429
Read K.bin: first values = -0.250920 0.593086 0.901429
Read V.bin: first values = -0.250920 0.593086 0.901429
free mem: 13278969856
totat mem: 15637086208
used mem:2248 MB
time:0.316737
==2610== Profiling application: ./cuda
==2610== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   94.91%  255.95ms         1  255.95ms  255.95ms  255.95ms  FlashAttn(float const *, float const *, float const *, float*, float*, float*)
                    4.14%  11.173ms         6  1.8622ms  45.151us  2.9053ms  [CUDA memcpy HtoD]
                    0.95%  2.5540ms         1  2.5540ms  2.5540ms  2.5540ms  [CUDA memcpy DtoH]
      API calls:   44.04%  266.82ms         6  44.470ms  10.030us  217.59ms  cudaHostAlloc
                   42.25%  255.97ms         1  255.97ms  255.97ms  255.97ms  cudaEventSynchronize
               

In [64]:
import torch
import numpy as np
import math
import torch.nn.functional as F

# Load shape
B, H, N, D = np.fromfile("shape.bin", dtype=np.int32)

# Load tensors
Q = torch.tensor(np.fromfile("Q.bin", dtype=np.float32)).reshape(B,H,N,D)
K = torch.tensor(np.fromfile("K.bin", dtype=np.float32)).reshape(B,H,N,D)
V = torch.tensor(np.fromfile("V.bin", dtype=np.float32)).reshape(B,H,N,D)

O_cuda = torch.tensor(np.fromfile("O_h.bin", dtype=np.float32)).reshape(B,H,N,D)

# Reference (FlashAttention math)
scores=torch.matmul(Q,K.transpose(-1,-2))
scores=scores/math.sqrt(D)
attn=torch.softmax(scores,dim=-1)
O_ref=torch.matmul(attn,V)

# Compare
diff = torch.abs(O_ref - O_cuda)

print("Max abs diff:", diff.max().item())
print("Mean diff:", diff.mean().item())

Max abs diff: 1.1175870895385742e-07
Mean diff: 7.96149191018003e-09


In [21]:
import torch
import time
import math
import torch.nn.functional as F

B = 8          # batch size
NH = 16        # number of heads
N = 1024       # sequence length (ONLY this as asked)
D = 64         # head dimension

M=32*1024
Bc = int(min(M // (4 * D), N))
Br = int(min(Bc, D))
Tc = (N + Bc - 1) // Bc
Tr = (N + Br - 1) // Br


device="cuda"

Q=torch.randn(B,NH,N,D,device=device,dtype=torch.float16)
K=torch.randn(B,NH,N,D,device=device,dtype=torch.float16)
V=torch.randn(B,NH,N,D,device=device,dtype=torch.float16)

def standard_attention(Q, K, V):
    scores=torch.matmul(Q,K.transpose(-1,-2))
    scores=scores/math.sqrt(D)
    # mask = torch.triu(torch.ones(scores.shape[-2:], device=Q.device), diagonal=1)
    # scores = scores.masked_fill(mask.bool(), float('-inf'))
    attn=torch.softmax(scores,dim=-1)
    O=torch.matmul(attn,V)
    return O

def torch_flash_attention(Q, K, V):
    O=F.scaled_dot_product_attention(Q,K,V,dropout_p=0.0,attn_mask=None,is_causal=False)
    return O

def python_flash_attention2(Q, K, V):
    O=torch.zeros_like(Q)
    row_full = torch.arange(N, device=Q.device)
    col_full = torch.arange(N, device=Q.device)
    for b in range(B):
        for h in range(NH):
            Qbh=Q[b,h]
            Kbh=K[b,h]
            Vbh=V[b,h]
            Obh=O[b,h]
            M=torch.full((N,1),-float('inf'),device=Q.device,dtype=Q.dtype)
            L=torch.zeros((N,1),device=Q.device,dtype=Q.dtype)
            for j in range(Tc):
                Kj=Kbh[j*Bc:(j+1)*Bc]
                Vj=Vbh[j*Bc:(j+1)*Bc]
                for i in range(Tr):
                    Qi=Qbh[i*Br:(i+1)*Br]

                    Sij=torch.matmul(Qi,Kj.T)
                    Sij=Sij/math.sqrt(D)

                    row_idx = row_full[i*Br:(i+1)*Br].unsqueeze(1)
                    col_idx = col_full[j*Bc:(j+1)*Bc].unsqueeze(0)
                    mask = col_idx > row_idx
                    Sij = Sij.masked_fill(mask, float('-inf'))
                    
                    m_i=torch.max(Sij,dim=1,keepdim=True).values
                    m_i = torch.where(torch.isinf(m_i), torch.zeros_like(m_i), m_i)
                    Pij=torch.exp(Sij-m_i)
                    Pij = torch.where(torch.isnan(Pij), torch.zeros_like(Pij), Pij)
                    l_i=torch.sum(Pij,dim=1,keepdim=True)

                    l_old=L[i*Br:(i+1)*Br]
                    m_old=M[i*Br:(i+1)*Br]
    
                    m_new=torch.max(m_old,m_i)
                    alpha=torch.exp(m_old-m_new)
                    beta=torch.exp(m_i-m_new)
                    
                    l_new=alpha*l_old+beta*l_i
                    O_i=Obh[i*Br:(i+1)*Br]
                    O_new=(alpha*l_old*O_i+beta*(Pij@Vj))/l_new

                    Obh[i*Br:(i+1)*Br]=O_new
                    M[i*Br:(i+1)*Br]=m_new
                    L[i*Br:(i+1)*Br]=l_new
            O[b,h]=Obh
    return O

def python_flash_attention(Q, K, V):
    O=torch.zeros_like(Q)
    row_full = torch.arange(N, device=Q.device)
    col_full = torch.arange(N, device=Q.device)
    M=torch.full((B,NH,N,1),-float('inf'),device=Q.device,dtype=Q.dtype)
    L=torch.zeros((B,NH,N,1),device=Q.device,dtype=Q.dtype)
    for j in range(Tc):
        Kj=K[:,:,j*Bc:(j+1)*Bc]
        Vj=V[:,:,j*Bc:(j+1)*Bc]
        for i in range(Tr):
            Qi=Q[:,:,i*Br:(i+1)*Br]

            Sij=torch.matmul(Qi,Kj.transpose(-1,-2))
            Sij=Sij/math.sqrt(D)
            
            m_i=torch.max(Sij,dim=-1,keepdim=True).values
            Pij=torch.exp(Sij-m_i)
            l_i=torch.sum(Pij,dim=-1,keepdim=True)

            l_old=L[:,:,i*Br:(i+1)*Br]
            m_old=M[:,:,i*Br:(i+1)*Br]

            m_new=torch.max(m_old,m_i)
            alpha=torch.exp(m_old-m_new)
            beta=torch.exp(m_i-m_new)
            
            l_new=alpha*l_old+beta*l_i
            O_i=O[:,:,i*Br:(i+1)*Br]
            O_new=(alpha*l_old*O_i+beta*(Pij@Vj))/l_new

            O[:,:,i*Br:(i+1)*Br]=O_new
            M[:,:,i*Br:(i+1)*Br]=m_new
            L[:,:,i*Br:(i+1)*Br]=l_new
    return O

torch.cuda.synchronize()

iters = 1
start = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    O = standard_attention(Q, K, V)
    torch.cuda.synchronize()

end = time.time()

avg_time_ms = (end - start) * 1000 / iters

print(f"Average Runtime for N=1024 under standard attention: {avg_time_ms:.3f} ms")
print("Output shape:", O.shape)
print()

torch.cuda.synchronize()

start1 = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    Ot = torch_flash_attention(Q, K, V)
    torch.cuda.synchronize()

end1 = time.time()

avg_time_ms1 = (end1 - start1) * 1000 / iters

print(f"Average Runtime for N=1024 under flash attention (torch): {avg_time_ms1:.3f} ms")
print("Output shape:", Ot.shape)
print()

torch.cuda.synchronize()

start2 = time.time()

for _ in range(iters):
    torch.cuda.synchronize()
    Ot2 = python_flash_attention(Q, K, V)
    torch.cuda.synchronize()

end2 = time.time()

avg_time_ms2 = (end2 - start2) * 1000 / iters

print(f"Average Runtime for N=1024 under flash attention (python1): {avg_time_ms2:.3f} ms")
print("Output shape:", Ot2.shape)
print()

torch.cuda.synchronize()

max_diff = (O - Ot).abs().max() 
print("Max absolute difference(torch vs pytorch):", max_diff.item())
max_diff2 = (O - Ot2).abs().max()
print("Max absolute difference(python vs torch):", max_diff2.item())
max_diff3 = (Ot - Ot2).abs().max()
print("Max absolute difference(pytorch vs python):", max_diff3.item())



Average Runtime for N=1024 under standard attention: 9.371 ms
Output shape: torch.Size([8, 16, 1024, 64])

Average Runtime for N=1024 under flash attention (torch): 5.020 ms
Output shape: torch.Size([8, 16, 1024, 64])

Average Runtime for N=1024 under flash attention (python1): 52.291 ms
Output shape: torch.Size([8, 16, 1024, 64])

Max absolute difference(torch vs pytorch): 0.000732421875
Max absolute difference(python vs torch): 0.0009765625
Max absolute difference(pytorch vs python): 0.0009765625


In [54]:
%%writefile main.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <fstream>

#define B 8
#define Nh 16
#define D 64
#define N 1024
#define Br 32
#define Bc 32

void read_file(const char* fname, float* data, int size) {
    std::ifstream file(fname, std::ios::binary);
    if (!file.is_open()) {
        printf("ERROR: cannot open %s\n", fname);  // ✅ catch missing file
        return;
    }
    file.read(reinterpret_cast<char*>(data), size * sizeof(float));
    printf("Read %s: first values = %f %f %f\n",  // ✅ confirm data loaded
           fname, data[0], data[1], data[2]);
    file.close();
 }

__global__ void FlashAttn(float *Q,float *K,float *V,float *m,float *l,float *O){
    extern __shared__ float smem[];
    float *Qi=smem;
    float *Kj=Qi+Br*D;
    float *Vj=Kj+Bc*D;
    float *Sij=Vj+Bc*D;

    int Tr = (N + Br - 1) / Br;
    int Tc = (N + Bc - 1) / Bc;

    int tx=threadIdx.x;
    int b=blockIdx.x;
    int h=blockIdx.y;

    int qkv_base=(b*Nh+h)*N*D;
    int lm_base=(b*Nh+h)*N;

    float scale=(1.0 / sqrtf(D));

    for(int j=0;j<Tc;j++){
        for(int x=0;x<D;x++){
            int kv_idx=qkv_base+(j*Bc+tx)*D+x;
            Kj[tx*D+x]=K[kv_idx];
            Vj[tx*D+x]=V[kv_idx];
        }
        __syncthreads();

        for(int i=0; i<Tr; i++){
            for(int x=0;x<D;x++){
                int q_idx=qkv_base+(i*Br+tx)*D+x;
                Qi[tx*D+x]=Q[q_idx];
            }
            int lm_idx=lm_base+(i*Br)+tx;
            float l_prev=l[lm_idx];
            float m_prev=m[lm_idx];

            float m_curr=-INFINITY;
            float l_curr=0.0f;
            for(int k=0;k<Bc;k++){
                float sum=0.0f;
                for(int x=0;x<D;x++){
                    float a=Qi[tx*D+x];
                    float b=Kj[k*D+x];
                    sum+=a*b;
                }
                float st=sum*scale;
                Sij[tx*Bc+k]=st;
                if(st>m_curr) m_curr=st;
            }
            for(int x=0;x<Bc;x++){
                Sij[tx*Bc+x]=exp(Sij[tx*Bc+x]-m_curr);
                l_curr+=Sij[tx*Bc+x];
            }

            float m_new=m_curr>m_prev? m_curr:m_prev;
            float alpha=exp(m_prev-m_new);
            float beta=exp(m_curr-m_new);
            float l_new=alpha*l_prev+beta*l_curr;

            for(int k=0;k<D;k++){
                float sum=0.0f;
                for(int x=0;x<Bc;x++){
                    float a=Sij[tx*Bc+x];
                    float b=Vj[x*D+k];
                    sum+=a*b;
                }
                int idx=qkv_base+(i*Br+tx)*D+k;
                O[idx]=(alpha*l_prev*O[idx]+beta*sum)/l_new;
            }
            l[lm_idx]=l_new;
            m[lm_idx]=m_new;
        }
        __syncthreads();
    }

}

int main(){

    //Allocating host memory
    float* Q_h,*K_h,*V_h,*O_h,*l_h,*m_h;
    cudaMallocHost(&Q_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&K_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&V_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&O_h,B*Nh*N*D*sizeof(float));
    cudaMallocHost(&l_h,B*Nh*N*sizeof(float));
    cudaMallocHost(&m_h,B*Nh*N*sizeof(float));

    // Read from files
    read_file("Q.bin", Q_h, B * Nh * N * D);
    read_file("K.bin", K_h, B * Nh * N * D);
    read_file("V.bin", V_h, B * Nh * N * D);

    //Initializing variables
    for(int i=0;i<B*Nh*N*D;i++){
        O_h[i] = 0;
    }
    for(int i=0;i<B*Nh*N;i++){
        l_h[i] = 0;
        m_h[i] = -INFINITY;
    }


    //Allocating device memory
    float* Q_d,*K_d,*V_d,*O_d,*l_d,*m_d;
    cudaMalloc(&Q_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&K_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&V_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&O_d,B*Nh*N*D*sizeof(float));
    cudaMalloc(&l_d,B*Nh*N*sizeof(float));
    cudaMalloc(&m_d,B*Nh*N*sizeof(float));

    //Memory Info
    size_t free_t,total_t;
    size_t used_mem;
    cudaMemGetInfo(&free_t,&total_t);
    used_mem = (total_t - free_t)/(1024*1024);

    printf("free mem: %zu\ntotat mem: %zu\nused mem:%zu MB\n",free_t,total_t,used_mem);

    //Copying to device
    cudaMemcpy(Q_d,Q_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(K_d,K_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(V_d,V_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(O_d,O_h,B*Nh*N*D*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(l_d,l_h,B*Nh*N*sizeof(float),cudaMemcpyHostToDevice);
    cudaMemcpy(m_d,m_h,B*Nh*N*sizeof(float),cudaMemcpyHostToDevice);

    //Launching the kernels

    dim3 grid(B,Nh);
    dim3 block(Br);
    size_t smem = (Br*D + 2*Bc*D + Br*Bc)*sizeof(float);

    cudaEvent_t total_start, total_stop;

    cudaEventCreate(&total_start);
    cudaEventCreate(&total_stop);
    cudaEventRecord(total_start);

    FlashAttn<<<grid,block,smem>>>(Q_d,K_d,V_d,m_d,l_d,O_d);

    cudaEventRecord(total_stop);
    cudaEventSynchronize(total_stop);
    float total_time;
    cudaEventElapsedTime(&total_time,total_start,total_stop);

    printf("time:%f\n",total_time/(1000)); //to get seconds for every iteration

    //copy to host
    cudaMemcpy(O_h, O_d, B*Nh*N*D*sizeof(float), cudaMemcpyDeviceToHost);

    std::ofstream out("O_h.bin", std::ios::binary);
    out.write(reinterpret_cast<char*>(O_h), B*Nh*N*D * sizeof(float));
    out.close();


    cudaFree(O_d);
    cudaFree(Q_d);
    cudaFree(K_d);
    cudaFree(V_d);
    cudaFree(m_d);
    cudaFree(l_d);
}

Overwriting main.cu


In [55]:
!nvcc main.cu -o main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [56]:
!nvprof ./main

==3117== NVPROF is profiling process 3117, command: ./main
Read Q.bin: first values = -0.250920 0.593086 0.901429
Read K.bin: first values = -0.250920 0.593086 0.901429
Read V.bin: first values = -0.250920 0.593086 0.901429
free mem: 13465616384
totat mem: 15637086208
used mem:2070 MB
time:0.320935
==3117== Profiling application: ./main
==3117== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   94.90%  251.46ms         1  251.46ms  251.46ms  251.46ms  FlashAttn(float*, float*, float*, float*, float*, float*)
                    4.13%  10.956ms         6  1.8260ms  45.023us  2.7178ms  [CUDA memcpy HtoD]
                    0.96%  2.5481ms         1  2.5481ms  2.5481ms  2.5481ms  [CUDA memcpy DtoH]
      API calls:   43.64%  266.50ms         6  44.416ms  5.1390us  219.08ms  cudaHostAlloc
                   41.18%  251.47ms         1  251.47ms  251.47ms  251.47ms  cudaEventSynchronize
                   11.38%  69.464ms 

In [57]:
import torch
import numpy as np
import math
import torch.nn.functional as F

# Load shape
B, H, N, D = np.fromfile("shape.bin", dtype=np.int32)

# Load tensors
Q = torch.tensor(np.fromfile("Q.bin", dtype=np.float32)).reshape(B,H,N,D)
K = torch.tensor(np.fromfile("K.bin", dtype=np.float32)).reshape(B,H,N,D)
V = torch.tensor(np.fromfile("V.bin", dtype=np.float32)).reshape(B,H,N,D)

O_cuda = torch.tensor(np.fromfile("O_h.bin", dtype=np.float32)).reshape(B,H,N,D)

# Reference (FlashAttention math)
scores=torch.matmul(Q,K.transpose(-1,-2))
scores=scores/math.sqrt(D)
attn=torch.softmax(scores,dim=-1)
O_ref=torch.matmul(attn,V)

# Compare
diff = torch.abs(O_ref - O_cuda)

print("Max abs diff:", diff.max().item())
print("Mean diff:", diff.mean().item())

Max abs diff: 1.1175870895385742e-07
Mean diff: 7.963901538232676e-09
